In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# ============================================================
# High-quality sinusoidal landscape with many local minima
# Shifted Rastrigin: global minimum at SHIFT, many local minima elsewhere
# ============================================================
BOUNDS = [-6.0, 6.0]            # 6x6 domain
SHIFT = np.array([5.0, 5.0])    # move global minimum to top-right corner

def rastrigin_shifted(X, A=10.0, noise=0.0):
    """
    Shifted Rastrigin objective. Global min at SHIFT with value ~0 (if noise=0).
    Many sinusoidal local minima across the domain.
    """
    Z = X - SHIFT
    x, y = Z[:, 0], Z[:, 1]
    # Classic Rastrigin per dimension
    val = (x**2 - A*np.cos(2*np.pi*x)) + (y**2 - A*np.cos(2*np.pi*y)) + 2*A
    if noise > 0:
        val = val + np.random.normal(scale=noise, size=val.shape)
    return val

# ============================================================
# Genetic Algorithm (slow, steady convergence)
# ============================================================
POP_SIZE = 120
GENS = 180
TOURN_SIZE = 2
CX_RATE = 0.5
MUT_RATE = 0.12
MUT_SCALE = 0.45
EXCLUDE_R = 2.0  # exclude initial points within this radius from SHIFT

rng = np.random.default_rng(1234)

def init_population():
    pop = []
    while len(pop) < POP_SIZE:
        c = rng.uniform(BOUNDS[0], BOUNDS[1], 2)
        if np.linalg.norm(c - SHIFT) > EXCLUDE_R:
            pop.append(c)
    return np.array(pop, dtype=float)

def select(pop, obj_vals):
    # Tournament selection minimizing objective
    chosen = []
    for _ in range(POP_SIZE):
        idx = rng.choice(POP_SIZE, TOURN_SIZE, replace=False)
        chosen.append(pop[idx[np.argmin(obj_vals[idx])]])
    return np.array(chosen)

def crossover(parents):
    kids = parents.copy()
    for i in range(0, POP_SIZE, 2):
        if rng.random() < CX_RATE:
            alpha = rng.random()
            p1, p2 = parents[i], parents[(i+1) % POP_SIZE]
            kids[i]   = alpha*p1 + (1-alpha)*p2
            kids[i+1] = (1-alpha)*p1 + alpha*p2
    return kids

def mutate(p):
    for i in range(POP_SIZE):
        if rng.random() < MUT_RATE:
            p[i] += rng.normal(scale=MUT_SCALE, size=2)
            p[i] = np.clip(p[i], BOUNDS[0], BOUNDS[1])
    return p

# Initialize population
pop = init_population()

# ============================================================
# Visualization setup: high-res sinusoidal contours + GA plot
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

# High-resolution background contours (static, no noise for crisp lines)
n_grid = 600
x = np.linspace(BOUNDS[0], BOUNDS[1], n_grid)
y = np.linspace(BOUNDS[0], BOUNDS[1], n_grid)
Xg, Yg = np.meshgrid(x, y)
Zg = rastrigin_shifted(np.c_[Xg.ravel(), Yg.ravel()], A=10.0, noise=0.0).reshape(Xg.shape)

# Contour lines and filled contours to emulate the tiled sinusoidal look
cont_f = ax1.contourf(Xg, Yg, Zg, levels=80, cmap='viridis')
cont_l = ax1.contour(Xg, Yg, Zg, levels=20, colors='k', linewidths=0.5, alpha=0.6)
cbar = fig.colorbar(cont_f, ax=ax1)
cbar.set_label('Objective value')

# Mark the true global minimum with a star
ax1.plot(SHIFT[0], SHIFT[1], marker='*', color='white', markersize=16,
         markeredgecolor='black', label='Global minimum')

# Scatter for population
scat = ax1.scatter(pop[:, 0], pop[:, 1], c='red', s=24, edgecolors='black', linewidths=0.3, zorder=5)

ax1.legend(loc='upper left')
ax1.set_xlim(BOUNDS); ax1.set_ylim(BOUNDS)
ax1.set_title('Population on sinusoidal landscape (shifted Rastrigin)')

# Best objective per generation plot
best_obj_history = []
line, = ax2.plot([], [], color='dodgerblue', lw=2.5)
ax2.set_xlim(0, GENS)
ax2.set_ylim(0, None)
ax2.set_xlabel('Generation')
ax2.set_ylabel('Best objective (lower is better)')
ax2.set_title('Best objective per generation')
ax2.grid(True, alpha=0.3)

# ============================================================
# Animation update
# ============================================================
def update(gen):
    global pop
    # Evaluate population on noisy surface to make convergence less trivial
    obj = rastrigin_shifted(pop, A=10.0, noise=0.8)
    best_obj = float(np.min(obj))
    best_obj_history.append(best_obj)

    # GA step
    parents = select(pop, obj)
    offspring = crossover(parents)
    pop = mutate(offspring)

    # Update visuals
    scat.set_offsets(pop)
    ax1.set_xlabel(f'Generation {gen+1}/{GENS}')
    xs = np.arange(len(best_obj_history))
    line.set_data(xs, best_obj_history)
    ax2.set_ylim(0, max(best_obj_history)*1.1)

    return scat, line

anim = animation.FuncAnimation(
    fig, update, frames=GENS, interval=220, blit=True
)

# Save as high-quality MP4 (ensure ffmpeg is available)
Writer = animation.writers['ffmpeg']
writer = Writer(fps=6, metadata=dict(artist='GA Sinusoidal Demo'), bitrate=6000)
anim.save('ga_sinusoidal_shifted_rastrigin_with_star.mp4', writer=writer)

plt.close(fig)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from tqdm.auto import tqdm  # progress bar

# ============================================================
# Domain and randomized global minimum (SHIFT)
# ============================================================
BOUNDS = [-6.0, 6.0]           # 6x6 domain
rng = np.random.default_rng()  # new seed each run -> new SHIFT each run

def sample_shift(bounds, margin=0.8):
    low, high = bounds
    return np.array([
        rng.uniform(low + margin, high - margin),
        rng.uniform(low + margin, high - margin)
    ], dtype=float)

SHIFT = sample_shift(BOUNDS, margin=0.8)  # random global minimum every run

# ============================================================
# Shifted Rastrigin objective (sinusoidal tiles with many local minima)
# ============================================================
def rastrigin_shifted(X, A=10.0, noise=0.0):
    Z = X - SHIFT
    x, y = Z[:, 0], Z[:, 1]
    val = (x**2 - A*np.cos(2*np.pi*x)) + (y**2 - A*np.cos(2*np.pi*y)) + 2*A
    if noise > 0:
        val = val + np.random.normal(scale=noise, size=val.shape)
    return val

# ============================================================
# Genetic Algorithm (slow, steady convergence)
# ============================================================
POP_SIZE = 120
GENS = 180
TOURN_SIZE = 2
CX_RATE = 0.5
MUT_RATE = 0.12
MUT_SCALE = 0.45
EXCLUDE_R = 2.0  # keep initial population away from the random SHIFT

def init_population():
    pop = []
    while len(pop) < POP_SIZE:
        c = rng.uniform(BOUNDS[0], BOUNDS[1], 2)
        if np.linalg.norm(c - SHIFT) > EXCLUDE_R:
            pop.append(c)
    return np.array(pop, dtype=float)

def select(pop, obj_vals):
    chosen = []
    for _ in range(POP_SIZE):
        idx = rng.choice(POP_SIZE, TOURN_SIZE, replace=False)
        chosen.append(pop[idx[np.argmin(obj_vals[idx])]])
    return np.array(chosen)

def crossover(parents):
    kids = parents.copy()
    for i in range(0, POP_SIZE, 2):
        if rng.random() < CX_RATE:
            alpha = rng.random()
            p1, p2 = parents[i], parents[(i+1) % POP_SIZE]
            kids[i]   = alpha*p1 + (1-alpha)*p2
            kids[i+1] = (1-alpha)*p1 + alpha*p2
    return kids

def mutate(p):
    for i in range(POP_SIZE):
        if rng.random() < MUT_RATE:
            p[i] += rng.normal(scale=MUT_SCALE, size=2)
            p[i] = np.clip(p[i], BOUNDS[0], BOUNDS[1])
    return p

# Initialize population (away from SHIFT)
pop = init_population()

# ============================================================
# Visualization: high-res contours + GA progress
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

# High-res background (noise-free for crisp contours)
n_grid = 600
x = np.linspace(BOUNDS[0], BOUNDS[1], n_grid)
y = np.linspace(BOUNDS[0], BOUNDS[1], n_grid)
Xg, Yg = np.meshgrid(x, y)
Zg = rastrigin_shifted(np.c_[Xg.ravel(), Yg.ravel()], A=10.0, noise=0.0).reshape(Xg.shape)

cont_f = ax1.contourf(Xg, Yg, Zg, levels=80, cmap='viridis')
cont_l = ax1.contour(Xg, Yg, Zg, levels=20, colors='k', linewidths=0.5, alpha=0.6)
cbar = fig.colorbar(cont_f, ax=ax1)
cbar.set_label('Objective value')

# Mark randomized global minimum with a star
ax1.plot(SHIFT[0], SHIFT[1], marker='*', color='white', markersize=16,
         markeredgecolor='black', label='Global minimum')

# Population scatter
scat = ax1.scatter(pop[:, 0], pop[:, 1], c='red', s=24, edgecolors='black', linewidths=0.3, zorder=5)

ax1.legend(loc='upper left')
ax1.set_xlim(BOUNDS); ax1.set_ylim(BOUNDS)
ax1.set_title('Population on sinusoidal landscape (shifted Rastrigin)')

# Best objective per generation plot
best_obj_history = []
line, = ax2.plot([], [], color='dodgerblue', lw=2.5)
ax2.set_xlim(0, GENS)
ax2.set_ylim(0, None)
ax2.set_xlabel('Generation')
ax2.set_ylabel('Best objective (lower is better)')
ax2.set_title('Best objective per generation')
ax2.grid(True, alpha=0.3)

# ============================================================
# tqdm progress bar
# ============================================================
pbar = tqdm(total=GENS, desc='Animating generations', leave=True)

def update(gen):
    global pop
    # Evaluate with noise to avoid trivial convergence
    obj = rastrigin_shifted(pop, A=10.0, noise=0.8)
    best_obj_history.append(float(np.min(obj)))

    # GA step
    parents = select(pop, obj)
    offspring = crossover(parents)
    pop = mutate(offspring)

    # Visual updates
    scat.set_offsets(pop)
    ax1.set_xlabel(f'Generation {gen+1}/{GENS}')
    xs = np.arange(len(best_obj_history))
    line.set_data(xs, best_obj_history)
    ax2.set_ylim(-1, max(best_obj_history)*1.1)

    # Advance progress bar
    pbar.update(1)
    if gen + 1 == GENS:
        pbar.close()

    return scat, line

anim = animation.FuncAnimation(
    fig, update, frames=GENS, interval=220, blit=True
)

# Save high-quality MP4 (requires ffmpeg) with tqdm during save
Writer = animation.writers['ffmpeg']
writer = Writer(fps=6, metadata=dict(artist='GA Sinusoidal Demo'), bitrate=6000)

# Wrap frame generation with tqdm for saving progress
with tqdm(total=GENS, desc='Saving MP4', leave=True) as save_pbar:
    def _progress_callback(i, n):
        save_pbar.update(1)
    anim.save('ga_sinusoidal_random_shift_with_star.mp4', writer=writer, progress_callback=_progress_callback)

plt.close(fig)
print("Randomized global minimum at:", SHIFT)


Animating generations:   0%|          | 0/180 [00:00<?, ?it/s]

Saving MP4:   0%|          | 0/180 [00:00<?, ?it/s]

Randomized global minimum at: [ 0.82159613 -3.16495868]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from tqdm.auto import tqdm  # progress bar

# ============================================================
# Domain and randomized global minimum (SHIFT)
# ============================================================
BOUNDS = [-6.0, 6.0]           # 6x6 domain
rng = np.random.default_rng()  # new seed each run -> new SHIFT each run

def sample_shift(bounds, margin=0.8):
    low, high = bounds
    return np.array([
        rng.uniform(low + margin, high - margin),
        rng.uniform(low + margin, high - margin)
    ], dtype=float)

SHIFT = sample_shift(BOUNDS, margin=0.8)  # random global minimum every run

# ============================================================
# Shifted Rastrigin objective (sinusoidal tiles with many local minima)
# ============================================================
def rastrigin_shifted(X, A=10.0, noise=0.0):
    Z = X - SHIFT
    x, y = Z[:, 0], Z[:, 1]
    val = (x**2 - A*np.cos(2*np.pi*(x/4))) + (y**2 - A*np.cos(2*np.pi*(y/4))) + 2*A
    if noise > 0:
        val = val + np.random.normal(scale=noise, size=val.shape)
    return val

# ============================================================
# Genetic Algorithm with Elitism
# ============================================================
POP_SIZE   = 120
GENS       = 180
TOURN_SIZE = 2
CX_RATE    = 0.5
MUT_RATE   = 0.12
MUT_SCALE  = 0.45
EXCLUDE_R  = 2.0     # keep initial population away from SHIFT
ELITE_K    = 3       # number of elites to carry forward unchanged

def init_population():
    pop = []
    while len(pop) < POP_SIZE:
        c = rng.uniform(BOUNDS[0], BOUNDS[1], 2)
        if np.linalg.norm(c - SHIFT) > EXCLUDE_R:
            pop.append(c)
    return np.array(pop, dtype=float)

def select(pop, obj_vals):
    chosen = []
    for _ in range(POP_SIZE):
        idx = rng.choice(POP_SIZE, TOURN_SIZE, replace=False)
        chosen.append(pop[idx[np.argmin(obj_vals[idx])]])
    return np.array(chosen)

def crossover(parents):
    kids = parents.copy()
    for i in range(0, POP_SIZE, 2):
        if rng.random() < CX_RATE:
            alpha = rng.random()
            p1, p2 = parents[i], parents[(i+1) % POP_SIZE]
            kids[i]   = alpha*p1 + (1-alpha)*p2
            kids[i+1] = (1-alpha)*p1 + alpha*p2
    return kids

def mutate(p):
    for i in range(p.shape[0]):
        if rng.random() < MUT_RATE:
            p[i] += rng.normal(scale=MUT_SCALE, size=2)
            p[i] = np.clip(p[i], BOUNDS[0], BOUNDS[1])
    return p

def inject_elites(parents, obj_vals):
    # Identify indices of the best ELITE_K individuals
    elite_idx = np.argsort(obj_vals)[:ELITE_K]
    elites = parents[elite_idx].copy()
    return elites

# Initialize population (away from SHIFT)
pop = init_population()

# ============================================================
# Visualization: high-res contours + GA progress
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

# High-res background (noise-free for crisp contours)
n_grid = 600
x = np.linspace(BOUNDS[0], BOUNDS[1], n_grid)
y = np.linspace(BOUNDS[0], BOUNDS[1], n_grid)
Xg, Yg = np.meshgrid(x, y)
Zg = rastrigin_shifted(np.c_[Xg.ravel(), Yg.ravel()], A=10.0, noise=0.0).reshape(Xg.shape)

cont_f = ax1.contourf(Xg, Yg, Zg, levels=80, cmap='viridis')
cont_l = ax1.contour(Xg, Yg, Zg, levels=20, colors='k', linewidths=0.5, alpha=0.6)
cbar = fig.colorbar(cont_f, ax=ax1)
cbar.set_label('Objective value')

# Mark randomized global minimum with a star
ax1.plot(SHIFT[0], SHIFT[1], marker='*', color='white', markersize=16,
         markeredgecolor='black', label='Global minimum')

# Population scatter
scat = ax1.scatter(pop[:, 0], pop[:, 1], c='red', s=24, edgecolors='black', linewidths=0.3, zorder=5)

ax1.legend(loc='upper left')
ax1.set_xlim(BOUNDS); ax1.set_ylim(BOUNDS)
ax1.set_title('Population on sinusoidal landscape (shifted Rastrigin)')

# Best objective per generation plot
best_obj_history = []
line, = ax2.plot([], [], color='dodgerblue', lw=2.5)
ax2.set_xlim(0, GENS)
ax2.set_ylim(0, None)
ax2.set_xlabel('Generation')
ax2.set_ylabel('Best objective (lower is better)')
ax2.set_title('Best objective per generation')
ax2.grid(True, alpha=0.3)

# ============================================================
# tqdm progress bar
# ============================================================
pbar = tqdm(total=GENS, desc='Animating generations', leave=True)

def update(gen):
    global pop
    # Evaluate with noise to avoid trivial convergence
    obj = rastrigin_shifted(pop, A=10.0, noise=0.8)
    best_obj_history.append(float(np.min(obj)))

    # Elitism: extract top-K elites before variation
    elites = inject_elites(pop, obj)

    # GA variation on the rest
    parents   = select(pop, obj)
    offspring = crossover(parents)
    offspring = mutate(offspring)

    # Replace worst K individuals in offspring with elites (steady-state elitism)
    # Rank by objective (lower better)
    off_obj = rastrigin_shifted(offspring, A=10.0, noise=0.0)  # noiseless for replacement decision
    worst_idx = np.argsort(off_obj)[-ELITE_K:]
    offspring[worst_idx] = elites

    pop = offspring

    # Visual updates
    scat.set_offsets(pop)
    ax1.set_xlabel(f'Generation {gen+1}/{GENS}')
    xs = np.arange(len(best_obj_history))
    line.set_data(xs, best_obj_history)
    ax2.set_ylim(-3, max(best_obj_history)*1.1)

    # Advance progress bar
    pbar.update(1)
    if gen + 1 == GENS:
        pbar.close()

    return scat, line

anim = animation.FuncAnimation(
    fig, update, frames=GENS, interval=220, blit=True
)

# Save high-quality MP4 (requires ffmpeg) with tqdm during save
Writer = animation.writers['ffmpeg']
writer = Writer(fps=6, metadata=dict(artist='GA Sinusoidal+Elitism'), bitrate=6000)

with tqdm(total=GENS, desc='Saving MP4', leave=True) as save_pbar:
    def _progress_callback(i, n):
        save_pbar.update(1)
    anim.save('ga_sinusoidal_random_shift_with_elitism.mp4', writer=writer, progress_callback=_progress_callback)

plt.close(fig)
print("Randomized global minimum at:", SHIFT)


Animating generations:   0%|          | 0/180 [00:00<?, ?it/s]

Saving MP4:   0%|          | 0/180 [00:00<?, ?it/s]

Randomized global minimum at: [-0.9973945  -3.82281406]


In [18]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from tqdm.auto import tqdm  # progress bars

# ============================
# Config switches
# ============================
use_elitism = True   # <--- toggle elitism on/off
ELITE_K = 3          # number of elites to carry if use_elitism=True

# ============================
# Domain and randomized global minimum (SHIFT)
# ============================
BOUNDS = [-6.0, 6.0]           # 6x6 domain
rng = np.random.default_rng()  # new seed each run -> new SHIFT each run

def sample_shift(bounds, margin=0.8):
    low, high = bounds
    return np.array([
        rng.uniform(low + margin, high - margin),
        rng.uniform(low + margin, high - margin)
    ], dtype=float)

SHIFT = sample_shift(BOUNDS, margin=0.8)  # random global minimum every run

# ============================
# Shifted Rastrigin objective (sinusoidal tiles)
# ============================
def rastrigin_shifted(X, A=10.0, noise=0.0, div=2):
    Z = X - SHIFT
    x, y = Z[:, 0], Z[:, 1]
    val = (x**2 - A*np.cos(2*np.pi*(x/div))) + (y**2 - A*np.cos(2*np.pi*(y/div))) + 2*A
    if noise > 0:
        val = val + np.random.normal(scale=noise, size=val.shape)
    return val

# ============================
# Genetic Algorithm setup
# ============================
POP_SIZE   = 120
GENS       = 180
TOURN_SIZE = 2
CX_RATE    = 0.5
MUT_RATE   = 0.12
MUT_SCALE  = 0.45
EXCLUDE_R  = 2.0  # keep initial population away from SHIFT

def init_population():
    pop = []
    while len(pop) < POP_SIZE:
        c = rng.uniform(BOUNDS[0], BOUNDS[1], 2)
        if np.linalg.norm(c - SHIFT) > EXCLUDE_R:
            pop.append(c)
    return np.array(pop, dtype=float)

def select(pop, obj_vals):
    chosen = []
    for _ in range(POP_SIZE):
        idx = rng.choice(POP_SIZE, TOURN_SIZE, replace=False)
        chosen.append(pop[idx[np.argmin(obj_vals[idx])]])
    return np.array(chosen)

def crossover(parents):
    kids = parents.copy()
    for i in range(0, POP_SIZE, 2):
        if rng.random() < CX_RATE:
            alpha = rng.random()
            p1, p2 = parents[i], parents[(i+1) % POP_SIZE]
            kids[i]   = alpha*p1 + (1-alpha)*p2
            kids[i+1] = (1-alpha)*p1 + alpha*p2
    return kids

def mutate(p):
    for i in range(p.shape[0]):
        if rng.random() < MUT_RATE:
            p[i] += rng.normal(scale=MUT_SCALE, size=2)
            p[i] = np.clip(p[i], BOUNDS[0], BOUNDS[1])
    return p

def get_elites(pop, obj_vals, k):
    elite_idx = np.argsort(obj_vals)[:k]
    return pop[elite_idx].copy()

# Initialize
pop = init_population()

# ============================
# Visualization (high quality)
# ============================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

# High-res, crisp contours (no noise in background)
n_grid = 600
x = np.linspace(BOUNDS[0], BOUNDS[1], n_grid)
y = np.linspace(BOUNDS[0], BOUNDS[1], n_grid)
Xg, Yg = np.meshgrid(x, y)
Zg = rastrigin_shifted(np.c_[Xg.ravel(), Yg.ravel()], A=10.0, noise=0.0).reshape(Xg.shape)

cont_f = ax1.contourf(Xg, Yg, Zg, levels=80, cmap='viridis')
ax1.contour(Xg, Yg, Zg, levels=20, colors='k', linewidths=0.5, alpha=0.6)
cbar = fig.colorbar(cont_f, ax=ax1)
cbar.set_label('Objective value')

# Mark randomized global minimum with a star
ax1.plot(SHIFT[0], SHIFT[1], marker='*', color='white', markersize=16,
         markeredgecolor='black', label='Global minimum')

# Population scatter
scat = ax1.scatter(pop[:, 0], pop[:, 1], c='red', s=24, edgecolors='black', linewidths=0.3, zorder=5)

ax1.legend(loc='upper left')
ax1.set_xlim(BOUNDS); ax1.set_ylim(BOUNDS)
ax1.set_title('Population on sinusoidal landscape (shifted Rastrigin)')

# Best objective plot
best_obj_history = []
line, = ax2.plot([], [], color='dodgerblue', lw=2.5)
ax2.set_xlim(0, GENS)
ax2.set_ylim(0, None)
ax2.set_xlabel('Generation')
ax2.set_ylabel('Best objective (lower is better)')
ax2.set_title('Best objective per generation')
ax2.grid(True, alpha=0.3)

# ============================
# tqdm progress bar
# ============================
pbar = tqdm(total=GENS, desc='Animating generations', leave=True)

def update(gen):
    global pop
    # Evaluate (noisy) objective for selection to keep dynamics interesting
    obj = rastrigin_shifted(pop, A=10.0, noise=0.8)
    best_obj_history.append(float(np.min(obj)))

    if use_elitism and ELITE_K > 0:
        elites = get_elites(pop, obj, ELITE_K)
    else:
        elites = None

    # Variation
    parents   = select(pop, obj)
    offspring = crossover(parents)
    offspring = mutate(offspring)

    if use_elitism and ELITE_K > 0:
        # Replace worst K in offspring with elites (ranked using noiseless eval)
        off_obj = rastrigin_shifted(offspring, A=10.0, noise=0.0)
        worst_idx = np.argsort(off_obj)[-ELITE_K:]
        offspring[worst_idx] = elites

    pop = offspring

    # Visual updates
    scat.set_offsets(pop)
    ax1.set_xlabel(f'Generation {gen+1}/{GENS}')
    xs = np.arange(len(best_obj_history))
    line.set_data(xs, best_obj_history)
    ax2.set_ylim(min(best_obj_history)-1, max(best_obj_history)*1.1)

    # Progress
    pbar.update(1)
    if gen + 1 == GENS:
        pbar.close()

    return scat, line

anim = animation.FuncAnimation(
    fig, update, frames=GENS, interval=220, blit=True
)

# Save high-quality MP4 (requires ffmpeg) with tqdm on save
Writer = animation.writers['ffmpeg']
writer = Writer(fps=6, metadata=dict(artist='GA Sinusoidal Demo'), bitrate=6000)

with tqdm(total=GENS, desc='Saving MP4', leave=True) as save_pbar:
    def _progress_callback(i, n):
        save_pbar.update(1)
    filename = f'./intelligent_systems_videos/ga_sinusoidal_random_shift_elitism_{use_elitism}.mp4'
    anim.save(filename, writer=writer, progress_callback=_progress_callback)

plt.close(fig)
print("Randomized global minimum at:", SHIFT, "| Elitism:", use_elitism, f"(K={ELITE_K})")


Animating generations:   0%|          | 0/180 [00:00<?, ?it/s]

Saving MP4:   0%|          | 0/180 [00:00<?, ?it/s]

Randomized global minimum at: [-5.03240094 -2.31655266] | Elitism: True (K=3)


In [14]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from tqdm.auto import tqdm  # progress bars

# ============================
# Domain and randomized global minimum (SHIFT)
# ============================
BOUNDS = [-6.0, 6.0]
rng = np.random.default_rng()

def sample_shift(bounds, margin=0.8):
    low, high = bounds
    return np.array([
        rng.uniform(low + margin, high - margin),
        rng.uniform(low + margin, high - margin)
    ], dtype=float)

SHIFT = sample_shift(BOUNDS, margin=0.8)

# ============================
# Shifted Rastrigin objective (sinusoidal tiles)
# ============================
def rastrigin_shifted(X, A=10.0, noise=0.0,div=2):
    Z = X - SHIFT
    x, y = Z[:, 0], Z[:, 1]
    val = (x**2 - A*np.cos(2*np.pi*(x/div))) + (y**2 - A*np.cos(2*np.pi*(y/div))) + 2*A
    if noise > 0:
        val += np.random.normal(scale=noise, size=val.shape)
    return val

# ============================
# PSO parameters
# ============================
SWARM_SIZE = 120
GENS       = 180
w          = 0.7    # inertia weight
c1, c2     = 1.5, 1.5  # cognitive & social coefficients

# Initialize swarm
positions = rng.uniform(BOUNDS[0], BOUNDS[1], size=(SWARM_SIZE, 2))
velocities = np.zeros_like(positions)

# Personal best
pbest_pos = positions.copy()
pbest_val = rastrigin_shifted(positions, noise=1.0)

# Global best
gbest_idx = np.argmin(pbest_val)
gbest_pos = pbest_pos[gbest_idx].copy()

# ============================
# Visualization setup
# ============================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,7))

# Contour background
n_grid = 600
x = np.linspace(BOUNDS[0], BOUNDS[1], n_grid)
y = np.linspace(BOUNDS[0], BOUNDS[1], n_grid)
Xg, Yg = np.meshgrid(x, y)
Zg = rastrigin_shifted(np.c_[Xg.ravel(), Yg.ravel()], noise=0.0).reshape(Xg.shape)
cf = ax1.contourf(Xg, Yg, Zg, levels=80, cmap='viridis')
ax1.contour(Xg, Yg, Zg, levels=20, colors='k', linewidths=0.5, alpha=0.6)
fig.colorbar(cf, ax=ax1, label='Objective')

# Mark global minimum
ax1.plot(SHIFT[0], SHIFT[1], '*', color='white', markersize=16,
         markeredgecolor='black', label='Global shift')
# Particle scatter
scat = ax1.scatter(positions[:,0], positions[:,1],
                   c='red', s=24, edgecolors='black', lw=0.3)
ax1.legend(loc='upper left')
ax1.set_xlim(BOUNDS); ax1.set_ylim(BOUNDS)
ax1.set_title('PSO Swarm on Sinusoidal Landscape')

# Best-objective plot
best_history = []
line, = ax2.plot([], [], color='dodgerblue', lw=2.5)
ax2.set_xlim(0, GENS)
ax2.set_ylim(0, None)
ax2.set_xlabel('Generation')
ax2.set_ylabel('Best objective')
ax2.set_title('Global Best Objective per Generation')
ax2.grid(True, alpha=0.3)

# Progress bar
pbar = tqdm(total=GENS, desc='Animating PSO', leave=True)

def update(gen):
    global positions, velocities, pbest_pos, pbest_val, gbest_pos

    # Evaluate
    vals = rastrigin_shifted(positions, noise=1.0)
    # Update personal bests
    better = vals < pbest_val
    pbest_pos[better] = positions[better]
    pbest_val[better] = vals[better]
    # Update global best
    idx = np.argmin(pbest_val)
    if pbest_val[idx] < rastrigin_shifted(gbest_pos[np.newaxis,:], noise=0.0):
        gbest_pos = pbest_pos[idx].copy()

    best_history.append(float(np.min(pbest_val)))

    r1, r2 = rng.random((SWARM_SIZE,1)), rng.random((SWARM_SIZE,1))
    # Velocity update
    velocities[:] = (w*velocities
                     + c1*r1*(pbest_pos - positions)
                     + c2*r2*(gbest_pos - positions))
    # Position update
    positions[:] += velocities
    # Enforce bounds
    positions[:] = np.clip(positions, BOUNDS[0], BOUNDS[1])

    # Visual updates
    scat.set_offsets(positions)
    ax1.set_xlabel(f'Gen {gen+1}/{GENS}')
    xs = np.arange(len(best_history))
    line.set_data(xs, best_history)
    ax2.set_ylim(min(best_history)-1, max(best_history)*1.1)

    pbar.update(1)
    if gen+1==GENS: pbar.close()
    return scat, line

anim = animation.FuncAnimation(
    fig, update, frames=GENS, interval=220, blit=True
)

# Save MP4 with a saving progress bar
Writer = animation.writers['ffmpeg']
writer = Writer(fps=6, metadata=dict(artist='PSO Demo'), bitrate=6000)
with tqdm(total=GENS, desc='Saving MP4', leave=True) as save_pbar:
    def prog(i, n): save_pbar.update(1)
    anim.save('./intelligent_systems_videos/pso_sinusoidal_demo.mp4', writer=writer, progress_callback=prog)

plt.close(fig)
print("Randomized global minimum at:", SHIFT)


Animating PSO:   0%|          | 0/180 [00:00<?, ?it/s]

Saving MP4:   0%|          | 0/180 [00:00<?, ?it/s]

Randomized global minimum at: [0.44605146 2.78019896]


In [15]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from tqdm.auto import tqdm

# ============================
# Domain and randomized global minimum (SHIFT)
# ============================
BOUNDS = [-6.0, 6.0]
rng = np.random.default_rng()

def sample_shift(bounds, margin=0.8):
    low, high = bounds
    return np.array([
        rng.uniform(low + margin, high - margin),
        rng.uniform(low + margin, high - margin)
    ], dtype=float)

SHIFT = sample_shift(BOUNDS, margin=0.8)

# ============================
# Shifted Rastrigin objective (sinusoidal tiles)
# ============================
def rastrigin_shifted(X, A=10.0, noise=0.0, div=2):
    Z = X - SHIFT
    x, y = Z[:,0], Z[:,1]
    val = (x**2 - A*np.cos(2*np.pi*(x/div))) + (y**2 - A*np.cos(2*np.pi*(y/div))) + 2*A
    if noise>0:
        val += np.random.normal(scale=noise, size=val.shape)
    return val

# ============================
# Differential Evolution parameters
# ============================
POP_SIZE = 120
GENS     = 180
F        = 0.8   # mutation factor
CR       = 0.9   # crossover probability

# Initialize population
pop = rng.uniform(BOUNDS[0], BOUNDS[1], size=(POP_SIZE, 2))

# ============================
# Visualization setup
# ============================
fig, (ax1, ax2) = plt.subplots(1,2, figsize=(14,7))

# Background contours
n_grid = 600
x = np.linspace(BOUNDS[0], BOUNDS[1], n_grid)
y = np.linspace(BOUNDS[0], BOUNDS[1], n_grid)
Xg, Yg = np.meshgrid(x,y)
Zg = rastrigin_shifted(np.c_[Xg.ravel(), Yg.ravel()], noise=0.0).reshape(Xg.shape)
cf = ax1.contourf(Xg, Yg, Zg, levels=80, cmap='viridis')
ax1.contour(Xg, Yg, Zg, levels=20, colors='k', linewidths=0.5, alpha=0.6)
fig.colorbar(cf, ax=ax1, label='Objective')

# Mark global minimum
ax1.plot(SHIFT[0], SHIFT[1], '*', color='white', markersize=16,
         markeredgecolor='black', label='Global shift')

# Scatter initial population
scat = ax1.scatter(pop[:,0], pop[:,1], c='orange', s=24, edgecolors='black', lw=0.3, zorder=5)
ax1.legend(loc='upper left')
ax1.set_xlim(BOUNDS); ax1.set_ylim(BOUNDS)
ax1.set_title('DE Population on Sinusoidal Landscape')

# Best-objective plot
best_history = []
line, = ax2.plot([], [], color='magenta', lw=2.5)
ax2.set_xlim(0, GENS)
ax2.set_ylim(0, None)
ax2.set_xlabel('Generation')
ax2.set_ylabel('Best objective')
ax2.set_title('Best Objective per Generation')
ax2.grid(True, alpha=0.3)

# Progress bar
pbar = tqdm(total=GENS, desc='Animating DE', leave=True)

def update(gen):
    global pop
    # Evaluate current population
    vals = rastrigin_shifted(pop, noise=1.0)
    best_history.append(float(np.min(vals)))

    new_pop = pop.copy()
    for i in range(POP_SIZE):
        # Mutation: DE/rand/1
        idxs = [idx for idx in range(POP_SIZE) if idx!=i]
        a,b,c = pop[rng.choice(idxs,3,replace=False)]
        donor = a + F*(b-c)
        donor = np.clip(donor, BOUNDS[0], BOUNDS[1])
        # Crossover
        cross = rng.random(2) < CR
        trial = np.where(cross, donor, pop[i])
        # Selection
        f_trial = rastrigin_shifted(trial[np.newaxis,:], noise=1.0)[0]
        if f_trial < vals[i]:
            new_pop[i] = trial
    pop[:] = new_pop

    # Visual updates
    scat.set_offsets(pop)
    ax1.set_xlabel(f'Gen {gen+1}/{GENS}')
    xs = np.arange(len(best_history))
    line.set_data(xs, best_history)
    ax2.set_ylim(min(best_history)-1, max(best_history)*1.1)

    pbar.update(1)
    if gen+1==GENS: pbar.close()
    return scat, line

anim = animation.FuncAnimation(fig, update, frames=GENS, interval=220, blit=True)

# Save MP4 with save progress bar
Writer = animation.writers['ffmpeg']
writer = Writer(fps=6, metadata=dict(artist='DE Demo'), bitrate=6000)
with tqdm(total=GENS, desc='Saving MP4', leave=True) as save_pbar:
    def prog(i,n): save_pbar.update(1)
    anim.save('./intelligent_systems_videos/de_sinusoidal_demo.mp4', writer=writer, progress_callback=prog)

plt.close(fig)
print("Randomized global minimum at:", SHIFT)


Animating DE:   0%|          | 0/180 [00:00<?, ?it/s]

Saving MP4:   0%|          | 0/180 [00:00<?, ?it/s]

Randomized global minimum at: [-4.146508    4.39099395]


In [16]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from tqdm.auto import tqdm

# ==========================================
# Domain and randomized global minimum SHIFT
# ==========================================
BOUNDS = [-6.0, 6.0]
rng = np.random.default_rng()

def sample_shift(bounds, margin=0.8):
    low, high = bounds
    return np.array([
        rng.uniform(low + margin, high - margin),
        rng.uniform(low + margin, high - margin)
    ], dtype=float)

SHIFT = sample_shift(BOUNDS, margin=0.8)

# ==========================================
# Shifted Rastrigin objective (sinusoidal)
# ==========================================
def rastrigin_shifted(X, A=10.0, noise=0.0, div=2):
    Z = X - SHIFT
    x, y = Z[:, 0], Z[:, 1]
    val = (x**2 - A*np.cos(2*np.pi*(x/div))) + (y**2 - A*np.cos(2*np.pi*(y/div))) + 2*A
    if noise > 0:
        val = val + np.random.normal(scale=noise, size=val.shape)
    return val

# ==========================================
# Simulated Annealing parameters
# ==========================================
T0               = 5.0      # initial temperature
T_MIN            = 1e-3    # final temperature
ALPHA            = 0.90    # cooling rate
N_ITER_PER_TEMP  = 200     # iterations per temperature
STEP_SIZE        = 1.0     # neighbor step size

# Initialize solution
current = rng.uniform(BOUNDS[0], BOUNDS[1], size=2)
current_val = rastrigin_shifted(current[np.newaxis,:], noise=1.0)[0]
best = current.copy()
best_val = current_val

# History for animation
positions = [current.copy()]
best_history = [best_val]

# ==========================
# Set up visualization
# ==========================
fig, (ax1, ax2) = plt.subplots(1,2, figsize=(14,7))

# Contour map
n_grid = 600
x = np.linspace(BOUNDS[0], BOUNDS[1], n_grid)
y = np.linspace(BOUNDS[0], BOUNDS[1], n_grid)
Xg, Yg = np.meshgrid(x,y)
Zg = rastrigin_shifted(np.c_[Xg.ravel(), Yg.ravel()], noise=0.0).reshape(Xg.shape)
cf = ax1.contourf(Xg, Yg, Zg, levels=80, cmap='viridis')
ax1.contour(Xg, Yg, Zg, levels=20, colors='k', linewidths=0.5, alpha=0.6)
fig.colorbar(cf, ax=ax1, label='Objective')

# Global SHIFT marker
ax1.plot(SHIFT[0], SHIFT[1], '*', color='white', markersize=16,
         markeredgecolor='black', label='True minimum')
scat_current = ax1.scatter([], [], c='orange', s=40, edgecolors='black', zorder=5)
scat_best    = ax1.scatter([], [], c='cyan',   s=60, edgecolors='black', zorder=6)
ax1.legend(loc='upper left')
ax1.set_xlim(BOUNDS); ax1.set_ylim(BOUNDS)
ax1.set_title('Simulated Annealing Walk')

# Best objective curve
line, = ax2.plot([], [], color='magenta', lw=2.5)
ax2.set_xlim(0,  int(np.log(T_MIN/T0)/np.log(ALPHA))*N_ITER_PER_TEMP )
ax2.set_ylim(0, None)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Best objective')
ax2.set_title('Best Objective over Time')
ax2.grid(True, alpha=0.3)

# Progress bar
total_iters = int(np.log(T_MIN/T0)/np.log(ALPHA))*N_ITER_PER_TEMP
pbar = tqdm(total=total_iters, desc='Annealing', leave=True)

# ==========================================
# Animation update function
# ==========================================
iters = 0
T = T0

def update(_):
    global current, current_val, best, best_val, T, iters

    # One temperature block
    for _ in range(N_ITER_PER_TEMP):
        # Propose neighbor
        candidate = current + rng.normal(scale=STEP_SIZE, size=2)
        candidate = np.clip(candidate, BOUNDS[0], BOUNDS[1])
        f_cand = rastrigin_shifted(candidate[np.newaxis, :], noise=1.0)[0]
        delta = f_cand - current_val

        # Metropolis acceptance
        if delta <= 0 or rng.random() < np.exp(-delta / T):
            current, current_val = candidate, f_cand
            if current_val < best_val:
                best, best_val = current.copy(), current_val

        # Record history
        positions.append(current.copy())
        best_history.append(best_val)
        iters += 1
        pbar.update(1)

    # Cool down
    T *= ALPHA

    # Update current and best scatters
    scat_current.set_offsets([positions[-1]])
    scat_best.set_offsets([best])

    # Update best‐objective line
    xs = np.arange(len(best_history))
    line.set_data(xs, best_history)
    ax2.set_ylim(min(best_history)-1, max(best_history)*1.1)

    return scat_current, scat_best, line


anim = animation.FuncAnimation(
    fig, update, frames=int(np.log(T_MIN/T0)/np.log(ALPHA)),
    interval=250, blit=True
)

# Save with progress
Writer = animation.writers['ffmpeg']
writer = Writer(fps=6, metadata=dict(artist='SA Demo'), bitrate=6000)
with tqdm(total=total_iters, desc='Saving MP4', leave=True) as save_pbar:
    def cb(i, n):
        save_pbar.update(N_ITER_PER_TEMP)
    anim.save('./intelligent_systems_videos/sa_sinusoidal_annealing.mp4', writer=writer, progress_callback=cb)

plt.close(fig)
pbar.close()
print("True minimum at", SHIFT)


Annealing:   0%|          | 0/16000 [00:00<?, ?it/s]

Saving MP4:   0%|          | 0/16000 [00:00<?, ?it/s]

True minimum at [-0.19066765 -4.41572223]
